# 01 · Basic Setup: Your First Calls to an LLM API

**Module 01 · Introduction to Agentic AI**

ChatGPT is a *product*. An **API** lets *your code* use the model behind it.
Every AI agent is built on API calls like the ones in this notebook, so this is where we start.

**What you will do**

| Step | Topic |
|---|---|
| 1 | Check Python, the virtual environment and your `.env` keys |
| 2 | Call **OpenAI**, **OpenRouter** and **Gemini** from Python |
| 3 | Understand **message roles** (`system` / `user` / `assistant`) |
| 4 | See why the API is **stateless** and why agents need **memory** |
| 5 | Measure **token usage and cost** |
| 6 | Play with **parameters** (`temperature`, `max_tokens`) |
| 7 | **Stream** a response token by token |

> You only need a key for one provider to follow along. Cells for providers without a key print a friendly hint and skip.

## 1 · Verify the Python environment

In [1]:
import sys

print("🐍 Python Environment Check")
print("=" * 40)
print(f"Python Version: {sys.version.split()[0]}")

major, minor = sys.version_info.major, sys.version_info.minor

# The course is written for Python 3.10 - 3.12 (the README says 3.10+).
if major == 3 and 10 <= minor <= 12:
    print(f"✅ Python {major}.{minor} is in the recommended range (3.10-3.12).")
else:
    print(f"⚠️ Python {major}.{minor} is outside the recommended range (3.10-3.12).")
    print("   This notebook still runs, but some packages in requirements.txt may not support it yet.")

🐍 Python Environment Check
Python Version: 3.13.1
⚠️ Python 3.13 is outside the recommended range (3.10-3.12).
   This notebook still runs, but some packages in requirements.txt may not support it yet.


## 2 · Set up a virtual environment

A virtual environment keeps this project's packages separate from the rest of your machine.
Run these commands **in a terminal** (not in the notebook), from the repo root:

In [2]:
# Create the virtual environment
#   python -m venv venv

# Activate it (Windows PowerShell)
#   .\venv\Scripts\Activate.ps1
# Activate it (macOS / Linux)
#   source venv/bin/activate

# Install the dependencies
#   pip install -r requirements.txt
# ...or, if you use uv:
#   uv pip install -r requirements.txt

## 3 · API keys and the `.env` file

An API key is a **password that is also a credit card**: anyone who has it can spend your money.
So keys never go inside your code or into Git. Instead:

1. Copy `.env.example` (committed to Git, contains placeholders only) to `.env`
   ```
   cp .env.example .env        # macOS / Linux
   copy .env.example .env      # Windows
   ```
2. Paste your real keys into `.env`. This file is listed in `.gitignore`, so it is never committed.
3. Load it in Python with `python-dotenv`.

The next cell checks your setup. It prints **names and ✅/❌ only, never the key values.**

In [3]:
import os
import importlib.util
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 1. requirements.txt (look in this folder and up to two levels above)
req = None
for path in [Path("requirements.txt"), Path("../requirements.txt"), Path("../../requirements.txt")]:
    if path.exists():
        req = path
        break

if req:
    print("✅ requirements.txt found")
else:
    print("❌ requirements.txt not found - are you in the right folder?")

# 2. .env file
env_path = find_dotenv()
if env_path:
    print("✅ .env file found")
else:
    print("❌ .env not found - copy .env.example to .env and add your keys")

# 3. Packages this notebook needs (pip name -> import name)
needed = {
    "python-dotenv": "dotenv",
    "requests": "requests",
    "langchain-openai": "langchain_openai",
    "langchain-google-genai": "langchain_google_genai",
}
for pip_name, module in needed.items():
    ok = importlib.util.find_spec(module) is not None
    if ok:
        print(f"✅ {pip_name}")
    else:
        print(f"❌ {pip_name}   ->  pip install -r requirements.txt")

# 4. Load the keys ONCE for the whole notebook
load_dotenv()
print("\n🔑 API keys found in .env:")
for name in ["OPENAI_API_KEY", "OPENROUTER_API_KEY", "GOOGLE_API_KEY"]:
    value = os.getenv(name)
    found = bool(value) and not value.startswith("your_")
    if found:
        print(f"   ✅ {name}")
    else:
        print(f"   ❌ {name}")

✅ requirements.txt found
✅ .env file found
✅ python-dotenv
✅ requests
✅ langchain-openai
✅ langchain-google-genai

🔑 API keys found in .env:
   ✅ OPENAI_API_KEY
   ✅ OPENROUTER_API_KEY
   ✅ GOOGLE_API_KEY


In [4]:
import os
import time
import requests
from IPython.display import Markdown, display

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


def have_key(value, name, url):
    """True if a real key is configured; otherwise print where to get one."""
    if value and not value.startswith("your_"):
        return True
    print(f"❌ {name} not found in .env")
    print(f"🔑 Get a key: {url}")
    print(f"📝 Add to .env: {name}=your_key_here")
    return False


print("✅ Environment loaded - ready to call some LLMs!")

✅ Environment loaded - ready to call some LLMs!


## 4 · Your first API calls

Three providers, three different libraries, **one identical pattern**:

```
API key  +  model name  +  messages  ->  API  ->  response (text + metadata)
```

| | Library | Why we show it |
|---|---|---|
| OpenAI | `langchain-openai` | A wrapper that gives every provider the same interface |
| OpenRouter | plain `requests` | The raw HTTP call that sits underneath every SDK |
| Gemini | `langchain-google-genai` | Same pattern, different provider |

### 4.1 OpenAI (via LangChain)
`ChatOpenAI` is a thin wrapper. `.invoke(...)` sends one request and waits for the full answer.

In [5]:
from langchain_openai import ChatOpenAI

print("🤖 TESTING OPENAI GPT")
print("=" * 35)

if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    try:
        llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0.3)

        start = time.time()
        response = llm.invoke("Describe the evolution of language models in 2 sentences.")
        elapsed = time.time() - start

        print("✅ OpenAI API working!")
        print(f"⏱️ Response time: {elapsed:.2f} seconds")
        print(f"🤖 Response: {response.text}")
    except Exception as e:
        print(f"❌ OpenAI error: {e}")

🤖 TESTING OPENAI GPT


✅ OpenAI API working!
⏱️ Response time: 1.58 seconds
🤖 Response: Language models have evolved from simple rule-based systems and n-gram models to complex neural networks, such as recurrent neural networks (RNNs) and transformers, which can capture intricate patterns in language. The introduction of large-scale pre-trained models like BERT and GPT has further advanced the field, enabling more nuanced understanding and generation of human-like text through transfer learning and fine-tuning techniques.


### 4.2 OpenRouter (raw HTTP with `requests`)
OpenRouter is a gateway: **one key, hundreds of models**. We call it with plain `requests` on purpose, to show that an "API call" is just an HTTP `POST` with three things:

* a **URL** (the endpoint),
* an **`Authorization` header** (your key),
* a **JSON body** (the model and your messages).

> ℹ️ The model below is a free-tier model. Free models come and go, so if you get a 404 swap in another one from <https://openrouter.ai/models?max_price=0>.

In [6]:
def ask_openrouter(prompt, model="nvidia/nemotron-3-ultra-550b-a55b:free", reasoning=True, max_tokens=None):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "reasoning": {"enabled": reasoning},
    }
    if max_tokens:
        payload["max_tokens"] = max_tokens

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]  # content (+ reasoning, if the model returns it)


print("🌐 TESTING OPENROUTER")
print("=" * 35)

if have_key(OPENROUTER_API_KEY, "OPENROUTER_API_KEY", "https://openrouter.ai/keys"):
    try:
        start = time.time()
        message = ask_openrouter("How many r's are in the word 'strawberry'?")
        elapsed = time.time() - start

        print("✅ OpenRouter API working!")
        print(f"⏱️ Response time: {elapsed:.2f} seconds")
        display(Markdown(message["content"]))   # renders the markdown nicely
    except requests.HTTPError as e:
        print(f"❌ OpenRouter HTTP error: {e}")
    except Exception as e:
        print(f"❌ OpenRouter error: {e}")

🌐 TESTING OPENROUTER


✅ OpenRouter API working!
⏱️ Response time: 2.61 seconds


There are **3** r's in "strawberry".

Here is the breakdown: st**r**awbe**rr**y

### 4.3 Google Gemini (via LangChain)
Gemini is **multimodal**: it can also take images, audio and video as input (we stay with text here).

Two things worth knowing:

* Newer Gemini models "think" before answering, and **thinking tokens count against `max_output_tokens`**. A cap that is too small (e.g. 200) can cut the answer off after a few words, so we use 1000.
* Gemini can return its answer as a *list of content blocks*. Use `response.text` to get the plain string.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("🎨 TESTING GOOGLE GEMINI")
print("=" * 35)

if have_key(GOOGLE_API_KEY, "GOOGLE_API_KEY", "https://aistudio.google.com/app/apikey"):
    try:
        llm = ChatGoogleGenerativeAI(
            google_api_key=GOOGLE_API_KEY,
            model="gemini-3.8-flash",
            temperature=0.3,
            max_output_tokens=1000,   # hard cap on generated tokens (thinking + answer)
        )

        start = time.time()
        response = llm.invoke("What makes Gemini different from other AI models? Answer in 3 sentences.")
        elapsed = time.time() - start

        print("✅ Google Gemini API working!")
        print(f"⏱️ Response time: {elapsed:.2f} seconds")
        print(f"🤖 Response: {response.text}")
    except Exception as e:
        print(f"❌ Google Gemini error: {e}")

🎨 TESTING GOOGLE GEMINI


✅ Google Gemini API working!
⏱️ Response time: 9.03 seconds
🤖 Response: Unlike many models that stitch together separate systems, Gemini was designed from the ground up to be natively multimodal, allowing it to seamlessly understand and reason across text, code, audio, images, and video. It also features an industry-leading long-context window, enabling it to ingest and analyze vast amounts of data—such as hours of video or millions of words—in a single prompt. Finally, it is deeply integrated with Google's ecosystem and infrastructure, providing real-time grounding with Google Search and high efficiency powered by custom TPU hardware.


## 5 · Message roles: `system`, `user`, `assistant`

An API call does not take "a question". It takes a **list of messages**, and every message has a **role**:

| Role | Who writes it | Purpose |
|---|---|---|
| `system` | You, the developer | Sets behaviour, tone and rules. **Hidden from the end user.** |
| `user` | The person (or your code) | The actual request |
| `assistant` | The model | Its previous replies |
| `tool` | Your code | The result of a tool the model asked to run (Module 02) |

In ChatGPT you only ever type `user` messages; OpenAI writes the `system` message for you.
With the API **you control it**, and it is the main lever for building a specific product or agent.

Below, the *same question* gets two very different answers purely because of the `system` message.

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0.3)
    question = "What is an API?"

    personas = {
        "🏴‍☠️ Pirate": "You are a pirate. Answer in at most 2 sentences.",
        "🧑‍💻 Strict engineer": "You are a terse senior engineer. Answer in exactly one precise sentence.",
    }

    for name, system_prompt in personas.items():
        messages = [
            SystemMessage(content=system_prompt),   # role = system
            HumanMessage(content=question),         # role = user
        ]
        reply = llm.invoke(messages)                # reply has role = assistant
        print(f"{name}\n   system: {system_prompt}\n   user:   {question}\n   🤖 {reply.text}\n")

🏴‍☠️ Pirate
   system: You are a pirate. Answer in at most 2 sentences.
   user:   What is an API?
   🤖 An API, or Application Programming Interface, be a set of rules that allows different software applications to communicate with each other. It be like a treasure map, guiding developers on how to access and use the features of a service or platform.



🧑‍💻 Strict engineer
   system: You are a terse senior engineer. Answer in exactly one precise sentence.
   user:   What is an API?
   🤖 An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and interact with each other.



## 6 · The API is stateless: it has no memory

This is the **biggest difference between ChatGPT and the API.**

* In **ChatGPT** you feel like the model "remembers" your conversation.
* In the **API** every call is independent. The model sees *only* the messages you send in that one request. Nothing is stored between calls.

ChatGPT's app quietly re-sends the whole conversation with every message. With the API, **that is your job.**

Watch it happen: three calls, same model.

In [9]:
if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0)

    # Call 1 - introduce ourselves
    intro = "Hi! My name is Alex and I am learning Agentic AI."
    reply_1 = llm.invoke(intro)
    print(f"👤 Call 1: {intro}\n🤖 {reply_1.text}\n")

    # Call 2 - a SEPARATE request with no history: the model has never heard of Alex
    reply_2 = llm.invoke("What is my name?")
    print(f"👤 Call 2 (no history): What is my name?\n🤖 {reply_2.text}\n")

    # Call 3 - we send the earlier turns ourselves, so the model can "remember"
    history = [
        HumanMessage(content=intro),
        AIMessage(content=reply_1.text),
        HumanMessage(content="What is my name?"),
    ]
    reply_3 = llm.invoke(history)
    print(f"👤 Call 3 (with history): What is my name?\n🤖 {reply_3.text}")

👤 Call 1: Hi! My name is Alex and I am learning Agentic AI.
🤖 Hi Alex! That's great to hear! Agentic AI is an exciting field. What specific aspects of Agentic AI are you interested in learning about?



👤 Call 2 (no history): What is my name?
🤖 I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. If you'd like to tell me your name, feel free!



👤 Call 3 (with history): What is my name?
🤖 Your name is Alex!


### Why do we need memory?

Call 2 failed and call 3 worked, and the only difference was **we re-sent the conversation**. That is all "memory" is at its simplest. It matters because:

1. **Models are stateless functions.** Same input in, same kind of output out. The model never "learns" from your call and keeps nothing between requests.
2. **Conversations need context.** "Fix *that* bug" or "make *it* shorter" only make sense if the model can see what *that* and *it* refer to.
3. **Agents work over many steps.** An agent that plans → searches → reads → decides has to remember what it already did, or it would repeat itself forever.
4. **But you can't just send everything every time.** Every re-sent message costs tokens (money) and the **context window** is limited. Long histories get slow, expensive and eventually don't fit.

So real systems manage memory deliberately:

| Type | What it is | Typical tool |
|---|---|---|
| **Short-term** | The recent messages in the current conversation | A list of messages, trimmed or summarised when it grows |
| **Long-term** | Facts and documents saved across sessions and retrieved when relevant | A vector database (RAG) |

You will build both in **Module 03 · Agent Memory & Retrieval**.

## 7 · Token usage and cost

ChatGPT costs a flat monthly fee. The API is **pay-as-you-go, billed per token**.

* A **token** is a chunk of text: roughly ¾ of an English word, or ~4 characters. "Agentic" might be 2 tokens.
* You pay for **input tokens** (everything you send, including system prompt and history) **and output tokens** (what the model writes). Output is usually several times more expensive.
* Every response tells you exactly how many tokens it used in `usage_metadata`, so you can calculate the cost of a call.

> 💲 The prices below are **example rates** for `gpt-4o-mini` (USD per 1 million tokens). Providers change prices often, so check <https://openai.com/api/pricing> before relying on them.

In [10]:
# Example rates in USD per 1,000,000 tokens - verify on the provider's pricing page
PRICE_PER_1M_INPUT = 0.15
PRICE_PER_1M_OUTPUT = 0.60

if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0.3)
    response = llm.invoke("Explain what a token is in an LLM, in about 60 words.")
    print(f"🤖 {response.text}\n")

    usage = response.usage_metadata          # {'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}
    cost_in = usage["input_tokens"] / 1_000_000 * PRICE_PER_1M_INPUT
    cost_out = usage["output_tokens"] / 1_000_000 * PRICE_PER_1M_OUTPUT

    print("📊 TOKEN USAGE")
    print(f"   Input tokens  : {usage['input_tokens']:>6}   -> ${cost_in:.6f}")
    print(f"   Output tokens : {usage['output_tokens']:>6}   -> ${cost_out:.6f}")
    print(f"   Total tokens  : {usage['total_tokens']:>6}   -> ${cost_in + cost_out:.6f}")
    print(f"\n💡 At this rate, 1,000 similar calls would cost about ${(cost_in + cost_out) * 1000:.2f}")

🤖 In the context of a large language model (LLM), a token is a unit of text that the model processes. Tokens can represent words, parts of words, or even punctuation marks. The model breaks down input text into these tokens to understand and generate language, enabling it to predict the next token based on the context provided by preceding tokens.

📊 TOKEN USAGE
   Input tokens  :     23   -> $0.000003
   Output tokens :     69   -> $0.000041
   Total tokens  :     92   -> $0.000045

💡 At this rate, 1,000 similar calls would cost about $0.04


**How to read this**

* Even a long-looking answer costs a tiny fraction of a cent, so experimenting is cheap.
* Cost scales with **tokens, not calls**. A 50-page document in the prompt costs far more than a one-line question.
* This is why memory (section 6) is a cost decision too: each extra message in the history is re-billed as input on *every* later call.
* Agents make many calls per task, so token tracking is how you keep them affordable. You will use it again in Module 05 (evals and observability).

## 8 · Parameters: controlling the model

ChatGPT gives you almost no knobs. The API gives you several. Two important ones:

| Parameter | What it does |
|---|---|
| `temperature` | Randomness. **0** = most predictable and consistent, **1+** = more varied and creative |
| `max_tokens` | Hard cap on the length of the answer (and therefore the cost) |

### 8.1 `temperature`: run the same prompt three times at each setting

In [11]:
if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    prompt = "Suggest a name for a coffee shop run by robots. Reply with just the name."

    for temp in (0.0, 1.2):
        llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=temp)
        print(f"🌡️ temperature = {temp}")
        for run in range(1, 4):
            print(f"   run {run}: {llm.invoke(prompt).text.strip()}")
        print()

🌡️ temperature = 0.0


   run 1: BrewBot Café


   run 2: BrewBot Café


   run 3: BrewBot Café

🌡️ temperature = 1.2


   run 1: Café Automata


   run 2: Robo Brews


   run 3: RoboBrew Café



At `temperature=0` the three runs are (nearly) identical, which is what you want for **extraction, classification and agent decisions**. At a high temperature they differ, which is what you want for **brainstorming and creative writing**.

### 8.2 `max_tokens`: a hard length limit
The model is cut off the moment it hits the limit, **even mid-sentence**. The response metadata tells you why it stopped: `stop` means it finished naturally, `length` means it was cut off.

In [12]:
if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    prompt = "Explain how the internet works in 2 sentences."

    for limit in (15, 200):
        llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0.3, max_tokens=limit)
        response = llm.invoke(prompt)
        print(f"✂️ max_tokens = {limit}")
        print(f"   finish_reason : {response.response_metadata['finish_reason']}")
        print(f"   output tokens : {response.usage_metadata['output_tokens']}")
        print(f"   answer        : {response.text}\n")

✂️ max_tokens = 15
   finish_reason : length
   output tokens : 15
   answer        : The internet is a global network of interconnected computers that communicate with each other using



✂️ max_tokens = 200
   finish_reason : stop
   output tokens : 56
   answer        : The internet is a global network of interconnected computers that communicate using standardized protocols, allowing data to be transmitted across various devices and networks. It relies on a system of servers, routers, and switches to route information, enabling users to access websites, send emails, and share data seamlessly.



## 9 · Streaming: tokens as they are generated

`.invoke()` waits until the **entire** answer is ready. That can take many seconds for long answers.

ChatGPT feels fast because it **streams**: the model sends tokens as it generates them and the UI prints each one immediately. With the API you opt in using `.stream()`, which yields small chunks.

The total time is about the same, but the **time to first token** is much shorter, and that is what makes an app *feel* responsive.

In [13]:
if have_key(OPENAI_API_KEY, "OPENAI_API_KEY", "https://platform.openai.com/api-keys"):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0.3)
    prompt = "Write a short, friendly 4-sentence introduction to Agentic AI."

    start = time.time()
    first_token_at = None
    chunks = 0

    print("🤖 ", end="")
    for chunk in llm.stream(prompt):
        if first_token_at is None:
            first_token_at = time.time() - start
        print(chunk.text, end="", flush=True)   # print each piece the moment it arrives
        chunks += 1
    total = time.time() - start

    print(f"\n\n⚡ Time to first token : {first_token_at:.2f}s")
    print(f"⏱️ Total time           : {total:.2f}s")
    print(f"📦 Chunks received      : {chunks}")

🤖 

Welcome

 to

 the

 world

 of

... (62 outputs left out)

-edge

 advancement

!



⚡ Time to first token : 0.70s
⏱️ Total time           : 1.47s
📦 Chunks received      : 74


## 🎯 Recap and what's next

You now know the building blocks behind every agent:

* ✅ **Keys** live in `.env`, never in code or Git
* ✅ Every provider follows the same pattern: key + model + messages -> response
* ✅ **Roles** (`system` / `user` / `assistant`) shape behaviour, and the `system` message is yours to write
* ✅ The API is **stateless**, so *you* send the history. That is what memory is
* ✅ You pay **per token**, and `usage_metadata` tells you exactly how many
* ✅ **Parameters** like `temperature` and `max_tokens` give you control
* ✅ **Streaming** makes responses feel instant

**Next → Module 02: Tool Calling & External Integrations.** Instead of only *talking*, the model will start *doing*: calling functions, searching the web and returning structured output. That is the first step from a chatbot to an agent. 🚀